In [5]:
import numpy as np
import json
import re

# Ler arquivos
with open("scenarios_pipeline.json", "r", encoding="utf-8") as f:
    pipeline = json.load(f)
with open("monitored_parameters.json", "r", encoding="utf-8") as f:
    param_info = json.load(f)

diagnosed_list = pipeline["diagnosed"]   # agora processa todos!
alfas = np.arange(0.0, 2.01, 0.01)

def limitar_range(x):
    return min(max(x, 0.0), 1.0)

def normalize_value(param, raw_val):
    min_val = param_info[param]["min_value"]
    max_val = param_info[param]["max_value"]
    if max_val == min_val:
        return 0.0
    return (raw_val - min_val) / (max_val - min_val)

def denormalize_value(param, norm_val):
    min_val = param_info[param]["min_value"]
    max_val = param_info[param]["max_value"]
    return norm_val * (max_val - min_val) + min_val

def extract_exprs(expr):
    """Extrai lista de tuplas (param, op, val) de uma expressão."""
    return re.findall(r"(p\d+)\s*(>=|<=|<|>)\s*([0-9.]+)", expr)

def build_expr(exprs):
    """Monta a expressão lógica a partir da lista de tuplas."""
    return " AND ".join([f"{var} {op} {val}" for (var, op, val) in exprs])

def perturba_expr(expr, alpha):
    exprs = extract_exprs(expr)
    out_exprs = []
    for var, op, val in exprs:
        val = float(val)
        # 1. Normalize
        val_norm = normalize_value(var, val)
        # 2. Perturbação na escala normalizada
        if op in ('>=', '>'):
            novo_norm = 1 - (1 - val_norm) * alpha
        elif op in ('<=', '<'):
            novo_norm = val_norm * alpha
        else:
            novo_norm = val_norm
        novo_norm = limitar_range(novo_norm)
        # 3. Denormalize antes de salvar
        val_real = denormalize_value(var, novo_norm)
        # Valor bonito: inteiro se for int, senão arredonda
        param_type = param_info[var].get("type", "float")
        if param_type == "int":
            val_real = int(round(val_real))
        else:
            val_real = round(val_real, 5)
        out_exprs.append((var, op, str(val_real)))
    return build_expr(out_exprs)

# Gera candidates para todos os diagnosed
candidates = []
for d_idx, diagnosed in enumerate(diagnosed_list):
    for idx, alpha in enumerate(alfas):
        scenario = {
            "name": f"diagnosed_{d_idx}_candidate_{idx}",
            "diagnosed_idx": d_idx,
            "diagnosed_name": diagnosed.get("name", f"diagnosed_{d_idx}"),
            "alpha": float(round(alpha, 4)),
            "given": perturba_expr(diagnosed["given"], alpha),
            "when": diagnosed["when"],
            "do": diagnosed["do"],
            "then": perturba_expr(diagnosed["then"], alpha)
        }
        candidates.append(scenario)

with open("shared_scenarios.json", "w", encoding="utf-8") as f:
    json.dump(candidates, f, indent=2, ensure_ascii=False)

print("Arquivo salvo: shared_scenarios.json")


Arquivo salvo: shared_scenarios.json


In [6]:
import json
import re

with open("shared_scenarios.json", "r", encoding="utf-8") as f:
    scenarios = json.load(f)
with open("monitored_parameters.json", "r", encoding="utf-8") as f:
    param_info = json.load(f)

all_params = list(param_info.keys())

def normalize(param, value):
    min_v = param_info[param]["min_value"]
    max_v = param_info[param]["max_value"]
    if max_v == min_v:
        return 0.0
    return (value - min_v) / (max_v - min_v)

def parse_clauses(expr):
    # Retorna lista de tuplas: (param, op, value)
    return re.findall(r"(p\d+)\s*(>=|<=|<|>)\s*([0-9.]+)", expr)

for scenario in scenarios:
    # Inicializa intervalos: [0, 1] para todos os parâmetros
    intervals = {param: [0.0, 1.0] for param in all_params}

    # Extrai todas as cláusulas de 'given' e 'then'
    clauses = []
    for bloco in ['given', 'then']:
        clauses += parse_clauses(scenario[bloco])

    for param, op, val in clauses:
        val = float(val)
        norm_val = normalize(param, val)
        if op in (">=", ">"):
            intervals[param][0] = norm_val  # mínimo permitido
        elif op in ("<=", "<"):
            intervals[param][1] = norm_val  # máximo permitido

    # Calcula volume (produto dos intervalos)
    volume = 1.0
    for min_v, max_v in intervals.values():
        intervalo = max(max_v - min_v, 0.0)
        volume *= intervalo
    scenario["volume"] = volume

with open("scenarios_with_norm_volume.json", "w", encoding="utf-8") as f:
    json.dump(scenarios, f, indent=2, ensure_ascii=False)

print("Arquivo salvo: scenarios_with_norm_volume.json")


Arquivo salvo: scenarios_with_norm_volume.json


In [7]:
import json
import pandas as pd

# 1. Carregar todos os campos do scenario_with_norm_volume.json
with open("scenarios_with_norm_volume.json", "r", encoding="utf-8") as f:
    candidates = json.load(f)
df_candidates = pd.DataFrame(candidates)

# 2. Ler similarity_results.jsonl e pegar diagnosed/candidate
sim_results = []
with open("similarity_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        sim_results.append({
            "diagnosed_given": data["diagnosed"]["given"],
            "diagnosed_then": data["diagnosed"]["then"],
            "candidate_given": data["candidate"]["given"],
            "candidate_then": data["candidate"]["then"],
            "similarity": data["similarity_result"],
            "name": data["candidate"]["name"]
        })
df_sim = pd.DataFrame(sim_results)

# 3. Merge pelos nomes e pega só as colunas desejadas
df_merged = pd.merge(
    df_sim,
    df_candidates[["name", "volume"]],
    on="name",
    how="inner"
)

# 4. Remover a coluna 'name' e ordenar as colunas como pedido
df_final = df_merged[[
    "diagnosed_given",
    "diagnosed_then",
    "candidate_given",
    "candidate_then",
    "volume",
    "similarity"
]]

df_final["similarity"] = df_final["similarity"] - (1/3)

pd.set_option('display.max_columns', None)
df_final

# Para salvar:
# df_final.to_csv("diagnosed_vs_candidate_only.csv", index=False)
# df_final.to_excel("diagnosed_vs_candidate_only.xlsx", index=False)


C:\Users\lucas_alves\AppData\Local\Temp\ipykernel_10372\464385714.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final["similarity"] = df_final["similarity"] - (1/3)


,diagnosed_given,diagnosed_then,candidate_given,candidate_then,volume,similarity
0,p0 >= 50 AND p1 >= 50 AND p2 >= 50,p4 >= 50 AND p5 >= 50 AND p6 >= 50,p0 >= 100.0 AND p1 >= 100.0 AND p2 >= 100.0,p4 >= 100.0 AND p5 >= 100.0 AND p6 >= 100.0,0.000000e+00,0.000000
1,p0 >= 50 AND p1 >= 50 AND p2 >= 50,p4 >= 50 AND p5 >= 50 AND p6 >= 50,p0 >= 99.5 AND p1 >= 99.5 AND p2 >= 99.5,p4 >= 99.5 AND p5 >= 99.5 AND p6 >= 99.5,1.562500e-14,0.006667
2,p0 >= 50 AND p1 >= 50 AND p2 >= 50,p4 >= 50 AND p5 >= 50 AND p6 >= 50,p0 >= 99.0 AND p1 >= 99.0 AND p2 >= 99.0,p4 >= 99.0 AND p5 >= 99.0 AND p6 >= 99.0,1.000000e-12,0.013333
3,p0 >= 50 AND p1 >= 50 AND p2 >= 50,p4 >= 50 AND p5 >= 50 AND p6 >= 50,p0 >= 98.5 AND p1 >= 98.5 AND p2 >= 98.5,p4 >= 98.5 AND p5 >= 98.5 AND p6 >= 98.5,1.139063e-11,0.020000
4,p0 >= 50 AND p1 >= 50 AND p2 >= 50,p4 >= 50 AND p5 >= 50 AND p6 >= 50,p0 >= 98.0 AND p1 >= 98.0 AND p2 >= 98.0,p4 >= 98.0 AND p5 >= 98.0 AND p6 >= 98.0,6.400000e-11,0.026667
...,...,...,...,...,...,...
799,p0 <= 50 AND p1 <= 50 AND p2 <= 50,p4 <= 50 AND p5 <= 50 AND p6 <= 50,p0 <= 98.0 AND p1 <= 98.0 AND p2 <= 98.0,p4 <= 98.0 AND p5 <= 98.0 AND p6 <= 98.0,8.858424e-01,0.340133
800,p0 <= 50 AND p1 <= 50 AND p2 <= 50,p4 <= 50 AND p5 <= 50 AND p6 <= 50,p0 <= 98.5 AND p1 <= 98.5 AND p2 <= 98.5,p4 <= 98.5 AND p5 <= 98.5 AND p6 <= 98.5,9.133083e-01,0.338407
801,p0 <= 50 AND p1 <= 50 AND p2 <= 50,p4 <= 50 AND p5 <= 50 AND p6 <= 50,p0 <= 99.0 AND p1 <= 99.0 AND p2 <= 99.0,p4 <= 99.0 AND p5 <= 99.0 AND p6 <= 99.0,9.414801e-01,0.336700
802,p0 <= 50 AND p1 <= 50 AND p2 <= 50,p4 <= 50 AND p5 <= 50 AND p6 <= 50,p0 <= 99.5 AND p1 <= 99.5 AND p2 <= 99.5,p4 <= 99.5 AND p5 <= 99.5 AND p6 <= 99.5,9.703725e-01,0.335007


In [8]:
import pandas as pd
import plotly.express as px

# Carregue o DataFrame já filtrado/organizado
df = df_final.copy()  # ou use o nome do DataFrame correto

fig = px.scatter(
    df,
    x="volume",
    y="similarity",
    color="similarity",  # ou "volume"
    hover_data=[
        "diagnosed_given", 
        "diagnosed_then", 
        "candidate_given", 
        "candidate_then"
    ],
    title="Scatter Plot 2D: Volume vs Similarity"
)

fig.update_traces(marker=dict(size=7, opacity=0.7))
fig.update_layout(xaxis_title="Volume", yaxis_title="Similarity")

fig.show()
